# Домашнее задание №14: Линейная регрессия с регуляризацией

### 1. Дайте определение регуляризации

Регуляризация — это приём борьбы с переобучением: к функции ошибок модели добавляют «штраф» за большие коэффициенты. Модель вынуждена искать баланс: хорошо подогнаться под данные, но не раздувать коэффициенты. Чем проще (компактнее) модель, тем устойчивее она работает на новых данных. Сила штрафа задаётся параметром (в sklearn обычно alpha).

### 2. L1 регуляризация

Штрафует сумму модулей коэффициентов. Главная фишка: умеет обнулять коэффициенты целиком — ненужные признаки просто выпадают из модели. Получается встроенный отбор признаков: модель остаётся «разреженной», работают только важные признаки.

### 3. L2 регуляризация

Штрафует сумму квадратов коэффициентов. Никого не обнуляет, но прижимает все коэффициенты к нулю и распределяет веса равномернее. Особенно полезна при мультиколлинеарности, где коэффициенты улетали в триллионы: L2 не даёт им раздуваться.

### 4. Практика

Собираем и нормализуем данные

In [65]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

data = pd.read_csv('./data/SouthGermanCredit_encoded.csv')
raw_X = data.iloc[:, :-1]  # все столбцы кроме последнего
y = data.iloc[:, -1]   # целевая переменная "Кредитный риск" (0:плохой, 1:хороший)
scaler = MinMaxScaler()
X = scaler.fit_transform(raw_X)

Обучаем: обычную линейную регрессию, L1, L2, ElasticNet

In [66]:
from sklearn.linear_model import LinearRegression, Lasso, Ridge, ElasticNet

model_linear_regression = LinearRegression().fit(X, y)
# Вставить альфу из следующего урока =)
model_l1 = Lasso(alpha=0.0003701957237807716).fit(X, y)
model_l2 = Ridge(alpha=4.852631378898325).fit(X, y)
model_ridge = ElasticNet(alpha=0.0007168684245366104).fit(X, y)

Смотрим полученные параметры разных моделей

In [67]:
print(pd.Series(model_linear_regression.coef_, index=raw_X.columns).sort_values(ascending=False).to_string(float_format='{:.6f}'.format))

семейное_положение_мужчина_женат_вдовец                    5157689670727.744141
семейное_положение_женщина_холоста                         5157689670727.702148
семейное_положение_женщина_несвободна_или_мужчина_холост   5157689670727.667969
семейное_положение_мужчина_разведён                        5157689670727.617188
срок_проживания_менее_1_года                               2315838709543.293945
срок_проживания_более_7_лет                                2315838709543.240234
срок_проживания_4_7_лет                                    2315838709543.235840
срок_проживания_1_4_года                                   2315838709543.181641
работа_безработный_нерезидент                              2019958849161.097656
работа_руководитель                                        2019958849161.036133
работа_неквалифицированный_резидент                        2019958849161.022949
работа_квалифицированный                                   2019958849161.013184
стаж_работы_4_7_лет                     

In [68]:
print(pd.Series(model_l1.coef_, index=raw_X.columns).sort_values(ascending=False).to_string(float_format='{:.30f}'.format))

поручители_поручитель                                       0.153282705148590270294306492360
гастарбайтер_да                                             0.149233148107039192975520336404
кредитная_история_все_кредиты_в_банке_выплачены             0.115375584407362050121115260026
цель_кредита_новое_авто                                     0.108793992614231574411753911136
статус_счёта_баланс_200_плюс                                0.090826454437378828510318840017
цель_кредита_отпуск                                         0.087525309685281604821227574575
ставка_платежа_более_35                                     0.076663656535177759909771566527
цель_кредита_бизнес                                         0.075260987343550522776780553613
сбережения_500_1000                                         0.070551241387946550531751199742
стаж_работы_4_7_лет                                         0.069167077235133336832184625109
возраст                                                     0.06698333

In [69]:
print(pd.Series(model_l2.coef_, index=raw_X.columns).sort_values(ascending=False).to_string(float_format='{:.6f}'.format))

статус_счёта_баланс_200_плюс                                0.140857
кредитная_история_все_кредиты_в_банке_выплачены             0.139275
поручители_поручитель                                       0.120863
цель_кредита_новое_авто                                     0.109829
цель_кредита_отпуск                                         0.090320
гастарбайтер_да                                             0.075617
цель_кредита_бизнес                                         0.073487
сбережения_500_1000                                         0.072253
стаж_работы_4_7_лет                                         0.070506
возраст                                                     0.069485
кредитная_история_выплачены_вовремя                         0.069422
семейное_положение_мужчина_женат_вдовец                     0.058442
ставка_платежа_более_35                                     0.056995
статус_счёта_баланс_0_200                                   0.052910
срок_проживания_менее_1_года      

In [70]:
print(pd.Series(model_ridge.coef_, index=raw_X.columns).sort_values(ascending=False).to_string(float_format='{:.30f}'.format))

поручители_поручитель                                       0.152278599966303912482956661734
статус_счёта_баланс_200_плюс                                0.133760745761775157181006079554
кредитная_история_все_кредиты_в_банке_выплачены             0.115297789177761664247690021057
цель_кредита_новое_авто                                     0.109178710646790216864943090513
цель_кредита_отпуск                                         0.086504756200648069697223263574
гастарбайтер_да                                             0.074542888060665016269901173018
цель_кредита_бизнес                                         0.073787145599376283144898991395
сбережения_500_1000                                         0.070260615867044948212871702253
стаж_работы_4_7_лет                                         0.069000102073781877098923587255
возраст                                                     0.066727608697133261594913733461
ставка_платежа_более_35                                     0.05932585

### Выводы

В первом случае мы получили гиперапараметры

Проверим качество обучения

In [71]:
print(model_linear_regression.score(X, y))
print(model_l1.score(X, y))
print(model_l2.score(X, y))
print(model_ridge.score(X, y))

0.28450994945707797
0.2837629717355378
0.2834241284776604
0.2837034652547442
